# LaStBeRu × Rubin reference catalog

This notebook runs the large-catalog workflow in separate stages:

1. **TAP coverage:** counts individual images and summarizes seeing, limiting magnitude, exposure time, and epochs by band.
2. **Coadd properties:** optionally samples coadd quality maps only for targets with individual-image coverage.
3. **Cutouts:** optionally loads full coadds only for the prioritized targets.

Run the first stage over all targets before enabling the more expensive stages.

In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from target_selection import (
    load_reference_catalog_config,
    run_reference_coverage_catalog,
    enrich_reference_catalog_coadds,
    generate_reference_catalog_cutouts,
)

_repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (parent for parent in _repo_candidates if (parent / "configs" / "lastberu_dp2.toml").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the target_selection repository or its notebooks/ directory.")

CONFIG_PATH = REPO_ROOT / "configs" / "lastberu_dp2.toml"
TARGET_LIMIT = 200  # Set None for all targets.
RUN_COVERAGE = True
RUN_COADD_PROPERTIES = False
RUN_CUTOUTS = False
MAX_CUTOUTS = 12

config = load_reference_catalog_config(CONFIG_PATH)
run_name = config.name if TARGET_LIMIT is None else f"{config.name}_preview_{TARGET_LIMIT}"
config = replace(
    config,
    name=run_name,
    target_limit=TARGET_LIMIT,
    max_cutouts=MAX_CUTOUTS,
    input_path=str(REPO_ROOT / config.input_path),
    output_dir=str(REPO_ROOT / config.output_dir),
    cache_dir=str(REPO_ROOT / config.cache_dir),
)
output_directory = Path(config.output_dir) / config.name
config

## 1. TAP coverage and individual-image summaries

This is the fast first pass. It does not initialize Butler and does not read coadd maps.

In [ ]:
coverage_path = output_directory / "coverage_catalog.csv"
if RUN_COVERAGE:
    coverage_catalog = run_reference_coverage_catalog(config)
elif coverage_path.exists():
    coverage_catalog = pd.read_csv(coverage_path)
else:
    raise FileNotFoundError(f"No coverage catalog at {coverage_path}. Set RUN_COVERAGE=True first.")

print(f"Coverage catalog: {len(coverage_catalog):,} targets")
coverage_catalog.head()

## Coverage summary

`n_images_<band>` counts unique target/detector visit rows. The seeing and `mag_lim` columns are medians of the corresponding individual-image values.

In [ ]:
bands = tuple(config.bands)
summary_rows = []
for band in bands:
    count_col = f"n_images_{band}"
    seeing_col = f"visit_seeing_mean_arcsec_{band}"
    maglim_col = f"visit_maglim_mean_{band}"
    counts = pd.to_numeric(coverage_catalog.get(count_col), errors="coerce").fillna(0)
    summary_rows.append({
        "band": band,
        "targets_with_images": int(counts.gt(0).sum()),
        "median_images_per_target": float(counts[counts.gt(0)].median()) if counts.gt(0).any() else 0.0,
        "median_seeing_arcsec": float(pd.to_numeric(coverage_catalog.get(seeing_col), errors="coerce").median()),
        "median_visit_maglim": float(pd.to_numeric(coverage_catalog.get(maglim_col), errors="coerce").median()),
    })
coverage_summary = pd.DataFrame(summary_rows)
print(f"Targets without images in any band: {int(pd.to_numeric(coverage_catalog['n_images_total'], errors='coerce').fillna(0).le(0).sum()):,}")
display(coverage_summary)
display(coverage_catalog[["target_id", "n_images_total", "visit_bands", "visit_query_status"]].head(20))

## 2. Optional coadd-property enrichment

This stage samples `maglim`, PSF size, exposure time, and sky-noise maps only for targets with individual-image coverage. It does not load full coadd images.

In [ ]:
reference_catalog_path = output_directory / "reference_catalog.csv"
if RUN_COADD_PROPERTIES:
    catalog = enrich_reference_catalog_coadds(config, coverage_catalog)
elif reference_catalog_path.exists():
    catalog = pd.read_csv(reference_catalog_path)
else:
    catalog = coverage_catalog.copy()

coadd_summary = pd.DataFrame({
    "metric": ["targets", "with coadd", "cutout candidates"],
    "value": [
        len(catalog),
        int(catalog.get("has_coadd", pd.Series(False, index=catalog.index)).fillna(False).sum()),
        int(catalog.get("cutout_eligible", pd.Series(False, index=catalog.index)).fillna(False).sum()),
    ],
})
display(coadd_summary)

## 3. Optional prioritized coadd cutouts

Full coadds are loaded only for the selected cutout budget.

In [ ]:
if RUN_CUTOUTS:
    if "cutout_selected" not in catalog:
        raise RuntimeError("Run the coadd-property stage first so cutout candidates are ranked.")
    catalog = generate_reference_catalog_cutouts(config, catalog)

if "cutout_status" in catalog:
    display(catalog["cutout_status"].value_counts(dropna=False).rename_axis("status").to_frame("targets"))
    generated = catalog.loc[catalog["cutout_status"].isin(["generated", "reused"])].sort_values("cutout_priority_rank")
    for row in generated.head(MAX_CUTOUTS).itertuples(index=False):
        print(f"Rank {int(row.cutout_priority_rank)} — {row.target_id}")
        display(Image(filename=row.cutout_path))